# 04 — TALAS layerwise CKA

Optional Figure A3. Compare only endpoint+$H_0$ against TALAS on one fixed held-out probe. The initialized and endpoint-only panels are removed to keep the diagnostic focused. Render this figure only when the contrast is stable across seeds/model pairs; CKA remains descriptive evidence rather than a causal claim.


In [ ]:
# 1. Cấu hình và artifact roots
from pathlib import Path
REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True
PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
TOPOLOGY_RUN_NAME = f"analysis_topology_{PAIR}_v1"
TALAS_RUN_ROOT = None  # None: tự tìm completed TALAS seeds bên dưới runs/
SEEDS, EPOCHS = [42, 43, 44], 5
PROBE_SIZE, ENCODE_BATCH = 512, 64
DEVICE = "cuda"


In [ ]:
# 2. Clone/fetch repo, dependencies và checkpoint discovery
import gc, subprocess, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoModel, AutoTokenizer
cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if AUTO_PULL_REPO:
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        print("[git] Bỏ qua pull vì repo có tracked changes.")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
from _analysis_common import PAIRS, final_checkpoint, set_paper_style
from src import structural_audit as audit
PAIR_CONFIG = PAIRS[PAIR]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
RUN_ROOT = PROJECT_DIR / "runs" / TOPOLOGY_RUN_NAME
OUT_DIR = RUN_ROOT / "layerwise_cka"
OUT_DIR.mkdir(parents=True, exist_ok=True)
combined_ckpts = {seed: final_checkpoint(RUN_ROOT / "combined_original" / f"seed_{seed}", EPOCHS) for seed in SEEDS}
talas_ckpts = {}
for seed in SEEDS:
    if TALAS_RUN_ROOT is None:
        candidates = list((PROJECT_DIR / "runs").glob(f"**/talas/seed_{seed}/checkpoint_epoch_{EPOCHS}.pt"))
        if not candidates: raise FileNotFoundError(f"No completed TALAS checkpoint for seed {seed}; set TALAS_RUN_ROOT.")
        talas_ckpts[seed] = max(candidates, key=lambda p: p.stat().st_mtime)
    else:
        talas_ckpts[seed] = final_checkpoint(Path(TALAS_RUN_ROOT) / f"seed_{seed}", EPOCHS)
set_paper_style()
print(f"Repo: {PROJECT_DIR} @ {git_head}")
for seed in SEEDS: print(f"seed {seed}: TALAS={talas_ckpts[seed]}")


In [ ]:
# 3. Encode the same fixed probe at every layer; cache tensors for cheap reruns
frame = pd.read_csv(TRAIN_DATA)
text_col = "text" if "text" in frame else "premise"
unique_texts = frame[text_col].astype(str).drop_duplicates()
texts = unique_texts.sample(n=min(PROBE_SIZE, len(unique_texts)), random_state=0).tolist()
def load_model(model_name, checkpoint=None, dtype=torch.float32):
    model = AutoModel.from_pretrained(model_name, torch_dtype=dtype)
    if checkpoint is not None:
        payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
        state = payload.get("model_state_dict", payload)
        missing, unexpected = model.load_state_dict(state, strict=False)
        bad = [name for name in missing if "layer" in name or "embeddings" in name]
        if unexpected or bad: raise RuntimeError(f"Checkpoint mismatch: missing={bad[:5]}, unexpected={unexpected[:5]}")
    return model.to(DEVICE).eval()
def encode_cached(label, model_name, checkpoint, pooling, dtype):
    path = OUT_DIR / f"{label}_layers.pt"
    if path.is_file(): return torch.load(path, map_location="cpu", weights_only=False)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = load_model(model_name, checkpoint, dtype=dtype)
    layers = audit.encode_texts(model, tokenizer, texts, device=DEVICE, pooling=pooling, batch_size=ENCODE_BATCH, layers=True, progress=True)["layers"]
    torch.save(layers, path)
    del model; gc.collect(); torch.cuda.empty_cache()
    return layers
teacher_layers = encode_cached("teacher", PAIR_CONFIG["teacher"], None, PAIR_CONFIG["teacher_pooling"], torch.bfloat16)
student_layers = {r"Endpoint + $H_0$": [], "TALAS": []}
for seed in SEEDS:
    student_layers[r"Endpoint + $H_0$"].append(encode_cached(f"endpoint_h0_seed_{seed}", PAIR_CONFIG["student"], combined_ckpts[seed], PAIR_CONFIG["student_pooling"], torch.float32))
    student_layers["TALAS"].append(encode_cached(f"talas_seed_{seed}", PAIR_CONFIG["student"], talas_ckpts[seed], PAIR_CONFIG["student_pooling"], torch.float32))


In [ ]:
# 4. Optional Figure A3 — Ours vs. TALAS teacher-layer × student-layer CKA
def cka_matrix(teacher, student):
    return np.asarray([[audit.linear_cka(t, s) for s in student] for t in teacher])
matrices_by_seed = {name: [cka_matrix(teacher_layers, layers) for layers in seed_layers] for name, seed_layers in student_layers.items()}
matrices = {name: np.mean(np.stack(seed_matrices), axis=0) for name, seed_matrices in matrices_by_seed.items()}
for name, seed_matrices in matrices_by_seed.items():
    safe_name = name.lower().replace(' ', '_').replace('$', '').replace('+', 'plus')
    for seed, matrix in zip(SEEDS, seed_matrices): pd.DataFrame(matrix).to_csv(OUT_DIR / f"cka_{safe_name}_seed_{seed}.csv", index=False)
fig, axes = plt.subplots(1, len(matrices), figsize=(5.5, 2.65), sharex=True, sharey=True)
for ax, (name, matrix) in zip(axes, matrices.items()):
    image = ax.imshow(matrix, origin="lower", aspect="auto", cmap="viridis", vmin=0, vmax=1, rasterized=True)
    ax.set(title=name, xlabel="student layer")
axes[0].set_ylabel("teacher layer")
fig.colorbar(image, ax=axes, label="linear CKA", shrink=.82, fraction=.035, pad=.03)
fig.text(.5, .03, "Mean CKA over three seeds on one shared held-out probe.", ha="center", fontsize=6.5, color="#6B7280")
fig.subplots_adjust(left=.10, right=.90, bottom=.22, top=.88, wspace=.12)
fig.savefig(OUT_DIR / "figure_A3_optional_talas_cka.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_A3_optional_talas_cka.png", dpi=300, bbox_inches="tight")
plt.show()
